# 02 · Silver layer — trusted, typed, validated tables

Silver is rebuilt **deterministically from Bronze** on every run (drop-and-recreate inside a transaction),
so rerunning never duplicates rows and Silver always reflects every Bronze load.

| Silver table | Grain | Built from |
|---|---|---|
| `silver.movies` | one row per movie | latest `bronze.movies_raw` snapshot |
| `silver.movie_genres` | one row per movie × genre | `silver.movies.genres` split on `\|` |
| `silver.ratings` | one row per user × movie (their latest rating) | all `bronze.ratings_raw` loads |
| `silver.movies_quarantine`, `silver.ratings_quarantine` | one row per rejected Bronze row + reason | — |
| `silver.quality_report` | one row per quality metric | — |

### Transformations and why

| Transformation | Why it is needed for the questions |
|---|---|
| Trim text and cast `userId`/`movieId` → `INTEGER`, `rating` → `DECIMAL(2,1)` | Bronze is text; numeric types are needed for joins and averages (all questions). `DECIMAL` keeps half-stars exact. |
| `timestamp` (Unix seconds, UTC) → `rated_at TIMESTAMP` and `rating_date DATE` | Q1 groups ratings by year; the Gold date dimension joins on the date. |
| Extract `release_year` from the trailing `(YYYY)` in the title; keep a `title_clean` without it | Q3 compares release decades; MovieLens has no separate year column. |
| Treat `(no genres listed)` as *no genres* (NULL) and split `genres` on `\|` into `silver.movie_genres` | Q2 ranks individual genres; a movie can have several. |
| De-duplicate on the key `(user_id, movie_id)`, keeping the latest rating | The source grain is "a user's current rating of a movie". Re-ratings or re-delivered rows must not be double-counted. |

No other derived columns are created — anything not used by Q1–Q4 is left out.

In [1]:
import pandas as pd
from IPython.display import display

from lakehouse import connect, rebuild_table

con = connect()

# Plausible time window for a MovieLens rating: the platform's data starts in 1995,
# and no rating can be later than the moment its batch was ingested.
MIN_RATING_DATE = "1995-01-01"

## 1. Movies

### Validation rules (movies)

| Rule | Check | Action |
|---|---|---|
| M1 | `movieId` is present and an integer | quarantine |
| M2 | `title` is present and non-empty | quarantine |
| M3 | `movieId` is unique in the snapshot | keep one row, quarantine the rest |
| M4 | parsed `release_year` lies between 1874 (earliest film) and the current year | keep the movie, set `release_year` to NULL (the movie is still valid for Q1, Q2 and Q4) |

In [2]:
con.execute(r'''
CREATE OR REPLACE TEMP TABLE movies_checked AS
WITH latest AS (           -- movies.csv is a full snapshot: use the most recent load only
    SELECT * FROM bronze.movies_raw
    WHERE load_id = (SELECT max(load_id) FROM bronze.movies_raw)
),
typed AS (
    SELECT movieId AS raw_movie_id, title AS raw_title, genres AS raw_genres,
           load_id, source_file,
           TRY_CAST(trim(movieId) AS INTEGER) AS movie_id,
           nullif(trim(title), '')            AS title,
           nullif(trim(genres), '')           AS genres
    FROM latest
)
SELECT *,
       CASE
           WHEN movie_id IS NULL THEN 'M1_missing_or_invalid_movie_id'
           WHEN title IS NULL    THEN 'M2_missing_title'
           WHEN row_number() OVER (PARTITION BY movie_id ORDER BY title, raw_genres) > 1
                                 THEN 'M3_duplicate_movie_id'
       END AS reject_reason
FROM typed
''')

n_movies = rebuild_table(con, "silver.movies", r'''
WITH parsed AS (
    SELECT movie_id, title,
           TRY_CAST(nullif(regexp_extract(title, '\((\d{4})(?:[-–]\d{4})?\)$', 1), '') AS INTEGER) AS parsed_year,
           CASE WHEN genres IS NULL OR genres = '(no genres listed)' THEN NULL ELSE genres END AS genres,
           load_id AS _source_load_id
    FROM movies_checked
    WHERE reject_reason IS NULL
)
SELECT movie_id,
       title,
       trim(regexp_replace(title, '\s*\((\d{4})(?:[-–]\d{4})?\)$', '')) AS title_clean,
       CASE WHEN parsed_year BETWEEN 1874 AND year(current_date) THEN parsed_year END AS release_year,
       genres,
       coalesce(len(string_split(genres, '|')), 0)::INTEGER AS genre_count,
       _source_load_id
FROM parsed
''')

n_movies_q = rebuild_table(con, "silver.movies_quarantine", '''
SELECT raw_movie_id, raw_title, raw_genres, load_id, source_file, reject_reason,
       current_timestamp::TIMESTAMP AS quarantined_at
FROM movies_checked WHERE reject_reason IS NOT NULL
''')

n_movie_genres = rebuild_table(con, "silver.movie_genres", '''
SELECT DISTINCT movie_id, trim(g) AS genre
FROM (SELECT movie_id, unnest(string_split(genres, '|')) AS g
      FROM silver.movies WHERE genres IS NOT NULL)
WHERE trim(g) <> ''
''')

print(f"silver.movies: {n_movies:,} | quarantined: {n_movies_q:,} | movie×genre rows: {n_movie_genres:,}")
display(con.sql("SELECT * FROM silver.movies ORDER BY movie_id LIMIT 5").df())

silver.movies: 62,423 | quarantined: 0 | movie×genre rows: 107,245


,movie_id,title,title_clean,release_year,genres,genre_count,_source_load_id
0,1,Toy Story (1995),Toy Story,1995,Adventure|Animation|Children|Comedy|Fantasy,5,load_0001
1,2,Jumanji (1995),Jumanji,1995,Adventure|Children|Fantasy,3,load_0001
2,3,Grumpier Old Men (1995),Grumpier Old Men,1995,Comedy|Romance,2,load_0001
3,4,Waiting to Exhale (1995),Waiting to Exhale,1995,Comedy|Drama|Romance,3,load_0001
4,5,Father of the Bride Part II (1995),Father of the Bride Part II,1995,Comedy,1,load_0001


In [3]:
# Year parsing check: titles where no release year could be extracted (kept, release_year = NULL)
display(con.sql('''
    SELECT count(*) AS movies_without_release_year,
           round(100.0 * count(*) / (SELECT count(*) FROM silver.movies), 2) AS pct_of_movies
    FROM silver.movies WHERE release_year IS NULL
''').df())
display(con.sql("SELECT movie_id, title FROM silver.movies WHERE release_year IS NULL LIMIT 5").df())

# Genre vocabulary after splitting
display(con.sql('''
    SELECT genre, count(*) AS movies FROM silver.movie_genres GROUP BY genre ORDER BY movies DESC
''').df().T)

,movies_without_release_year,pct_of_movies
0,411,0.66


,movie_id,title
0,144726,Man Crazy
1,149334,Nocturnal Animals
2,151517,Beauty and the Breast
3,151693,Holy Hell
4,151955,Veljet


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18
genre,Drama,Comedy,Thriller,Romance,Action,Horror,Documentary,Crime,Adventure,Sci-Fi,Children,Animation,Mystery,Fantasy,War,Western,Musical,Film-Noir,IMAX
movies,25606,16870,8654,7719,7348,5989,5605,5319,4145,3595,2935,2929,2925,2731,1874,1399,1054,353,195


## 2. Ratings

### Validation rules (ratings)

Each Bronze row is tested in order; the first failed rule becomes its `reject_reason` and the row goes to
`silver.ratings_quarantine` with its raw values, `load_id` and source file, so it can be traced back.

| Rule | Check | Action |
|---|---|---|
| R1 | `userId` and `movieId` are present integers | quarantine |
| R2 | `rating` is numeric, between 0.5 and 5.0, in half-star steps | quarantine |
| R3 | `timestamp` is an integer between 1995-01-01 and the batch's `ingested_at` (no future ratings) | quarantine |
| R4 | `movieId` exists in `silver.movies` (referential integrity) | quarantine |
| R5 | `(user_id, movie_id)` is unique — duplicates are classified as *exact duplicates* or *superseded by a newer rating*; the latest is kept | quarantine the extras |

Step 1 types and classifies every Bronze row into a staging table; step 2 splits it into the trusted table and
the quarantine table; the staging table is then dropped.

In [4]:
con.execute("DROP TABLE IF EXISTS silver.ratings_stage")
con.execute(f'''
CREATE TABLE silver.ratings_stage AS
WITH typed AS (
    SELECT userId AS raw_user_id, movieId AS raw_movie_id, rating AS raw_rating,
           "timestamp" AS raw_timestamp, source_file, load_id, ingested_at,
           TRY_CAST(trim(userId) AS INTEGER)    AS user_id,
           TRY_CAST(trim(movieId) AS INTEGER)   AS movie_id,
           TRY_CAST(trim(rating) AS DOUBLE)     AS rating_num,
           TRY_CAST(trim("timestamp") AS BIGINT) AS rating_epoch
    FROM bronze.ratings_raw
),
checked AS (
    SELECT t.*,
           CASE
               WHEN t.user_id IS NULL OR t.movie_id IS NULL THEN 'R1_missing_or_invalid_id'
               WHEN t.rating_num IS NULL                     THEN 'R2_rating_not_numeric'
               WHEN t.rating_num < 0.5 OR t.rating_num > 5.0
                    OR t.rating_num * 2 <> round(t.rating_num * 2) THEN 'R2_rating_out_of_domain'
               WHEN t.rating_epoch IS NULL                   THEN 'R3_timestamp_invalid'
               WHEN t.rating_epoch < epoch(TIMESTAMP '{MIN_RATING_DATE}')
                    OR t.rating_epoch > epoch(t.ingested_at) THEN 'R3_timestamp_out_of_range'
               WHEN m.movie_id IS NULL                       THEN 'R4_orphan_movie_id'
           END AS reject_reason
    FROM typed t
    LEFT JOIN silver.movies m ON t.movie_id = m.movie_id
),
ranked AS (
    SELECT *,
           row_number()  OVER w AS key_rank,
           first_value(rating_epoch) OVER w AS kept_epoch,
           first_value(rating_num)   OVER w AS kept_rating
    FROM checked
    WINDOW w AS (PARTITION BY reject_reason IS NULL, user_id, movie_id
                 ORDER BY rating_epoch DESC, load_id DESC, rating_num DESC)
)
SELECT * EXCLUDE (reject_reason, key_rank, kept_epoch, kept_rating),
       coalesce(reject_reason,
                CASE WHEN key_rank > 1 AND rating_epoch = kept_epoch AND rating_num = kept_rating
                         THEN 'R5_exact_duplicate'
                     WHEN key_rank > 1 THEN 'R5_superseded_by_newer_rating'
                END) AS reject_reason
FROM ranked
''')

n_ratings = rebuild_table(con, "silver.ratings", '''
SELECT user_id,
       movie_id,
       CAST(rating_num AS DECIMAL(2,1))                 AS rating,
       epoch_ms(rating_epoch * 1000)                    AS rated_at,     -- UTC
       CAST(epoch_ms(rating_epoch * 1000) AS DATE)      AS rating_date,  -- UTC calendar day
       load_id     AS _source_load_id,
       source_file AS _source_file
FROM silver.ratings_stage
WHERE reject_reason IS NULL
''')

n_ratings_q = rebuild_table(con, "silver.ratings_quarantine", '''
SELECT raw_user_id, raw_movie_id, raw_rating, raw_timestamp, load_id, source_file,
       reject_reason, current_timestamp::TIMESTAMP AS quarantined_at
FROM silver.ratings_stage
WHERE reject_reason IS NOT NULL
''')

reason_counts = con.sql('''
    SELECT reject_reason, count(*) AS rows FROM silver.ratings_stage
    WHERE reject_reason IS NOT NULL GROUP BY 1 ORDER BY 1
''').df()
con.execute("DROP TABLE silver.ratings_stage")

print(f"silver.ratings: {n_ratings:,} valid rows | quarantined: {n_ratings_q:,}")
display(reason_counts if len(reason_counts) else pd.DataFrame({"reject_reason": ["(none)"], "rows": [0]}))
display(con.sql("SELECT * FROM silver.ratings LIMIT 5").df())
display(con.sql("DESCRIBE silver.ratings").df()[["column_name", "column_type"]])

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

silver.ratings: 25,000,095 valid rows | quarantined: 0


,reject_reason,rows
0,(none),0


,user_id,movie_id,rating,rated_at,rating_date,_source_load_id,_source_file
0,13699,6620,4.0,2005-02-23 02:13:58,2005-02-23,load_0001,ratings_1995_2014.csv
1,13700,3285,3.5,2016-08-31 09:20:25,2016-08-31,load_0002,ratings_2015_2019.csv
2,13700,3978,3.5,2004-01-22 10:23:00,2004-01-22,load_0001,ratings_1995_2014.csv
3,13700,5890,3.0,2004-01-22 08:44:15,2004-01-22,load_0001,ratings_1995_2014.csv
4,13700,8638,3.5,2008-10-14 16:46:45,2008-10-14,load_0001,ratings_1995_2014.csv


,column_name,column_type
0,user_id,INTEGER
1,movie_id,INTEGER
2,rating,"DECIMAL(2,1)"
3,rated_at,TIMESTAMP
4,rating_date,DATE
5,_source_load_id,VARCHAR
6,_source_file,VARCHAR


## 3. Quality report

The required evidence in one table: Bronze vs Silver vs quarantine counts, duplicates, nulls in critical
columns (raw Bronze and typed Silver), and key uniqueness. It is also stored as `silver.quality_report`
so every run leaves an auditable record.

In [5]:
con.execute('''
CREATE OR REPLACE TEMP TABLE quality_now AS
WITH
b AS (
    SELECT count(*) AS bronze_rows,
           count(*) FILTER (WHERE userId IS NULL OR trim(userId) = '')           AS null_userId,
           count(*) FILTER (WHERE movieId IS NULL OR trim(movieId) = '')         AS null_movieId,
           count(*) FILTER (WHERE rating IS NULL OR trim(rating) = '')           AS null_rating,
           count(*) FILTER (WHERE "timestamp" IS NULL OR trim("timestamp") = '') AS null_timestamp
    FROM bronze.ratings_raw
),
s AS (
    SELECT count(*) AS silver_rows,
           count(*) FILTER (WHERE user_id IS NULL)   AS null_user_id,
           count(*) FILTER (WHERE movie_id IS NULL)  AS null_movie_id,
           count(*) FILTER (WHERE rating IS NULL)    AS null_rating,
           count(*) FILTER (WHERE rated_at IS NULL)  AS null_rated_at,
           count(*) - (SELECT count(*) FROM (SELECT DISTINCT user_id, movie_id FROM silver.ratings)) AS dup_keys
    FROM silver.ratings
),
q AS (
    SELECT count(*) AS quarantined,
           count(*) FILTER (WHERE reject_reason LIKE 'R5%') AS duplicates
    FROM silver.ratings_quarantine
),
mb AS (SELECT count(*) AS n FROM bronze.movies_raw
       WHERE load_id = (SELECT max(load_id) FROM bronze.movies_raw)),
ms AS (SELECT count(*) AS n,
              count(*) - count(DISTINCT movie_id) AS dup_keys,
              count(*) FILTER (WHERE release_year IS NULL) AS no_year,
              count(*) FILTER (WHERE genres IS NULL) AS no_genres,
              count(*) - count(DISTINCT title) AS repeated_titles
       FROM silver.movies),
mq AS (SELECT count(*) AS n FROM silver.movies_quarantine)
SELECT * FROM (VALUES
  ('ratings', 'Bronze row count',                         (SELECT bronze_rows FROM b)),
  ('ratings', 'Silver valid row count',                   (SELECT silver_rows FROM s)),
  ('ratings', 'Quarantined row count',                    (SELECT quarantined FROM q)),
  ('ratings', 'Duplicate rows (R5, quarantined)',         (SELECT duplicates FROM q)),
  ('ratings', 'Bronze nulls/empties: userId',             (SELECT null_userId FROM b)),
  ('ratings', 'Bronze nulls/empties: movieId',            (SELECT null_movieId FROM b)),
  ('ratings', 'Bronze nulls/empties: rating',             (SELECT null_rating FROM b)),
  ('ratings', 'Bronze nulls/empties: timestamp',          (SELECT null_timestamp FROM b)),
  ('ratings', 'Silver nulls: user_id',                    (SELECT null_user_id FROM s)),
  ('ratings', 'Silver nulls: movie_id',                   (SELECT null_movie_id FROM s)),
  ('ratings', 'Silver nulls: rating',                     (SELECT null_rating FROM s)),
  ('ratings', 'Silver nulls: rated_at',                   (SELECT null_rated_at FROM s)),
  ('ratings', 'Silver duplicate keys (user_id, movie_id)',(SELECT dup_keys FROM s)),
  ('movies',  'Bronze row count (latest snapshot)',       (SELECT n FROM mb)),
  ('movies',  'Silver valid row count',                   (SELECT n FROM ms)),
  ('movies',  'Quarantined row count',                    (SELECT n FROM mq)),
  ('movies',  'Silver duplicate keys (movie_id)',         (SELECT dup_keys FROM ms)),
  ('movies',  'Movies with no parseable release year',    (SELECT no_year FROM ms)),
  ('movies',  'Movies with no genres listed',             (SELECT no_genres FROM ms)),
  ('movies',  'Titles shared by >1 movie_id (kept, informational)', (SELECT repeated_titles FROM ms))
) AS t(entity, metric, value)
''')

con.execute('''
CREATE TABLE IF NOT EXISTS silver.quality_report (
    run_at TIMESTAMP, entity VARCHAR, metric VARCHAR, value BIGINT
)''')
con.execute("INSERT INTO silver.quality_report SELECT current_timestamp::TIMESTAMP, * FROM quality_now")

metrics = con.sql("SELECT * FROM quality_now").df()
display(metrics)

,entity,metric,value
0,ratings,Bronze row count,25000095
1,ratings,Silver valid row count,25000095
2,ratings,Quarantined row count,0
3,ratings,"Duplicate rows (R5, quarantined)",0
4,ratings,Bronze nulls/empties: userId,0
5,ratings,Bronze nulls/empties: movieId,0
6,ratings,Bronze nulls/empties: rating,0
7,ratings,Bronze nulls/empties: timestamp,0
8,ratings,Silver nulls: user_id,0
9,ratings,Silver nulls: movie_id,0


In [6]:
# Hard checks: the run fails loudly if Silver cannot be trusted.
m = dict(zip(metrics["metric"] + " | " + metrics["entity"], metrics["value"]))
bronze_r = m["Bronze row count | ratings"]
silver_r = m["Silver valid row count | ratings"]
quar_r = m["Quarantined row count | ratings"]

assert bronze_r == silver_r + quar_r, "Bronze != Silver + quarantine (rows lost)"
assert m["Silver duplicate keys (user_id, movie_id) | ratings"] == 0, "ratings key not unique"
assert m["Silver duplicate keys (movie_id) | movies"] == 0, "movies key not unique"
assert m["Silver nulls: rating | ratings"] == 0 and m["Silver nulls: rated_at | ratings"] == 0

print(f"Reconciliation: {bronze_r:,} Bronze = {silver_r:,} Silver + {quar_r:,} quarantined  ✔")
print("Key (user_id, movie_id) unique in silver.ratings  ✔")
print("Key movie_id unique in silver.movies  ✔")

Reconciliation: 25,000,095 Bronze = 25,000,095 Silver + 0 quarantined  ✔
Key (user_id, movie_id) unique in silver.ratings  ✔
Key movie_id unique in silver.movies  ✔


In [7]:
con.close()
print("Silver layer complete.")

Silver layer complete.
